In [ ]:
from typing import Tuple, Union
import math
import struct

Number = Union[int, float, complex, Tuple[float, float]]
ComplexPair = Tuple[float, float]

def _as_complex(value: Number) -> complex:
    if isinstance(value, tuple):
        if len(value) != 2:
            raise ValueError("Complex pairs must have exactly two elements.")
        return complex(float(value[0]), float(value[1]))
    return complex(value)

def _as_float(value: Number) -> float:
    if isinstance(value, tuple):
        if len(value) != 2:
            raise ValueError("Real inputs must have exactly two elements.")
        if abs(float(value[1])) > 1e-12:
            raise ValueError("pade_sqrt accepts real-valued inputs only.")
        return float(value[0])
    if isinstance(value, complex):
        if abs(value.imag) > 1e-12:
            raise ValueError("pade_sqrt accepts real-valued inputs only.")
        return float(value.real)
    return float(value)

def _to_pair(value: complex) -> ComplexPair:
    return (float(value.real), float(value.imag))

def _sqrt_initial_guess(value: float) -> float:
    if value == 0.0:
        return 0.0
    if value < 0.0:
        raise ValueError("pade_sqrt accepts non-negative real inputs only.")

    bits = struct.unpack(">I", struct.pack(">f", float(value)))[0]
    exponent_bits = (bits >> 23) & 0xFF

    if exponent_bits == 0:
        mantissa = float(value)
        exponent = 0
        while mantissa < 1.0:
            mantissa *= 2.0
            exponent -= 1
        guess_exponent = (exponent + 127) // 2
    elif exponent_bits == 0xFF:
        raise ValueError("pade_sqrt does not support NaN or infinity.")
    else:
        guess_exponent = (exponent_bits + 127) // 2

    return math.ldexp(1.0, int(guess_exponent) - 127)

def _scale_exp_approx(exp_bits: int) -> int:
    return ((exp_bits + 2 * 127) * 0b1011) >> 5

def _cbrt_initial_guess(value: complex) -> complex:
    s = _as_complex(value)
    if s == 0:
        return 0j

    re = float(s.real)
    im = float(s.imag)

    # Work with raw 32-bit float bitpatterns
    re_bits = struct.unpack(">I", struct.pack(">f", re))[0]
    im_bits = struct.unpack(">I", struct.pack(">f", im))[0]

    # Extract sign bit (0x80000000) and exponent (bits 23..30)
    re_sign_bit = re_bits & 0x80000000
    im_sign_bit = im_bits & 0x80000000

    e_re = (re_bits >> 23) & 0xFF
    e_im = (im_bits >> 23) & 0xFF

    if e_re == 0xFF or e_im == 0xFF:
        raise ValueError("pade_cbrt does not support NaN or infinity.")

    # Treat subnormals by promoting exponent to 1 when magnitude non-zero
    if e_re == 0 and re != 0.0:
        e_re = 1
    if e_im == 0 and im != 0.0:
        e_im = 1

    # Use the larger exponent to get a conservative seed magnitude
    e_max = max(e_re, e_im)

    # Compute approximate cube-root exponent: floor((E-127)/3)+127
    e_cbrt = ((e_max - 127) // 3) + 127

    # Clamp into valid exponent range for IEEE-754 single precision
    if e_cbrt <= 0:
        e_cbrt = 1
    elif e_cbrt >= 0xFF:
        e_cbrt = 0xFE

    # Build seed bitpatterns with zero mantissa and chosen exponent,
    # preserving sign bits from original inputs.
    re_seed_bits = re_sign_bit | (e_cbrt << 23)
    im_seed_bits = im_sign_bit | (e_cbrt << 23)

    re_seed = struct.unpack(">f", struct.pack(">I", re_seed_bits))[0]
    im_seed = struct.unpack(">f", struct.pack(">I", im_seed_bits))[0]

    # If original component was exactly zero, keep seed exactly zero
    if re == 0.0:
        re_seed = 0.0
    if im == 0.0:
        im_seed = 0.0
    return complex(re_seed, im_seed)

def pade_sqrt(z: Number, iterations: int = 5) -> ComplexPair:
    s = _as_float(z)
    if s == 0.0:
        return (0.0, 0.0)

    p = _sqrt_initial_guess(s)
    for _ in range(max(1, int(iterations))):
        p2 = p * p
        denominator = 3.0 * p2 + s
        if abs(denominator) < 1e-18:
            break
        p = p * (p2 + 3.0 * s) / denominator

    return (float(p), 0.0)

def pade_cbrt(z: Number, iterations: int = 5) -> ComplexPair:
    s = _as_complex(z)
    if s == 0:
        return (0.0, 0.0)

    p = _cbrt_initial_guess(s)
    for _ in range(max(1, int(iterations))):
        p3 = p * p * p
        denominator = 2.0 * p3 + s
        if abs(denominator) < 1e-18:
            break
        p = p * (p3 + 2.0 * s) / denominator

    return _to_pair(p)


In [10]:
import random

# ==========================================
# KHU VỰC CHẠY TEST (TESTBENCH)
# ==========================================
def run_benchmark(num_tests=10000, sqrt_iterations=10, cbrt_iterations=10):
    random.seed(7)

    # 500 test số thực dương cho sqrt
    real_samples = [random.uniform(0.0, 1000.0) for _ in range(num_tests // 2)]

    # 500 test số phức cho cbrt
    complex_samples = [complex(random.uniform(-500.0, 500.0), random.uniform(-500.0, 500.0)) for _ in range(num_tests // 2)]

    sqrt_errors = []
    cbrt_errors = []
    threshold = 1e-3

    sqrt_pass = 0
    cbrt_pass = 0

    for s in real_samples:
        my_sqrt = complex(*pade_sqrt(float(s), iterations=sqrt_iterations))
        err_sqrt = abs(my_sqrt**2 - s) / s if s != 0 else 0.0
        sqrt_errors.append(err_sqrt)
        if err_sqrt < threshold:
            sqrt_pass += 1

    for z in complex_samples:
        my_cbrt = complex(*pade_cbrt(z, iterations=cbrt_iterations))
        err_cbrt = abs(my_cbrt**3 - z) / (abs(z) if z != 0 else 1.0)
        cbrt_errors.append(err_cbrt)
        if err_cbrt < threshold:
            cbrt_pass += 1

    print(f"Tổng số case sqrt đã test: {len(real_samples)}")
    print(f"Tổng số case cbrt đã test: {len(complex_samples)}")
    print(f"\n--- KẾT QUẢ CĂN BẬC 2 ({sqrt_iterations} lần lặp) ---")
    print(f"Số test đạt (sai số tương đối < {threshold}): {sqrt_pass}/{len(real_samples)}")
    print(f"Sai số trung bình: {sum(sqrt_errors)/len(sqrt_errors):.6f}")
    print(f"Sai số lớn nhất : {max(sqrt_errors):.6f}")

    print(f"\n--- KẾT QUẢ CĂN BẬC 3 ({cbrt_iterations} lần lặp) ---")
    print(f"Số test đạt (sai số tương đối < {threshold}): {cbrt_pass}/{len(complex_samples)}")
    print(f"Sai số trung bình: {sum(cbrt_errors)/len(cbrt_errors):.6f}")
    print(f"Sai số lớn nhất : {max(cbrt_errors):.6f}")

# Chạy thử
run_benchmark(10000, sqrt_iterations=16, cbrt_iterations=16)

Tổng số case sqrt đã test: 5000
Tổng số case cbrt đã test: 5000

--- KẾT QUẢ CĂN BẬC 2 (16 lần lặp) ---
Số test đạt (sai số tương đối < 0.001): 5000/5000
Sai số trung bình: 0.000000
Sai số lớn nhất : 0.000000

--- KẾT QUẢ CĂN BẬC 3 (16 lần lặp) ---
Số test đạt (sai số tương đối < 0.001): 5000/5000
Sai số trung bình: 0.000000
Sai số lớn nhất : 0.000000


In [11]:
EPS = complex(-0.5, 0.8660254037844386)

def solve_cubic(a, b, c, d, sqrt_iterations=5, cbrt_iterations=5):
    b_n = b / (-3.0 * a)
    c_n = c / (-3.0 * a)
    d_n = d / (-3.0 * a)

    delta_0 = b_n**2 + c_n
    delta_1 = 2.0 * b_n**3 + 3.0 * b_n * c_n + 3.0 * d_n
    delta = delta_1**2 - 4.0 * delta_0**3

    roots = []

    if delta.real < 0:
        sqrt_delta_imag = pade_sqrt(abs(float(delta.real)), iterations=sqrt_iterations)[0]
        sqrt_delta = complex(0.0, sqrt_delta_imag)
        C_val = complex(*pade_cbrt((delta_1 + sqrt_delta) / 2.0, iterations=cbrt_iterations))
        C_conj = C_val.conjugate()
        for k in range(3):
            roots.append(b_n + (EPS**k) * C_val + (EPS**(2 * k)) * C_conj)
    elif abs(4.0 * delta_0**3) < 1e-9 and delta_1 <= 0:
        C_val = complex(*pade_cbrt(delta_1, iterations=cbrt_iterations))
        for k in range(3):
            roots.append(b_n + (EPS**k) * C_val)
    else:
        sqrt_delta = complex(*pade_sqrt(float(delta.real), iterations=sqrt_iterations))
        C_val = complex(*pade_cbrt((delta_1 + sqrt_delta) / 2.0, iterations=cbrt_iterations))
        for k in range(3):
            roots.append(b_n + (EPS**k) * C_val + (EPS**(2 * k)) * (delta_0 / C_val))

    return roots

In [12]:
import random
import numpy as np

def sort_roots(roots):
    return sorted([complex(r) for r in roots], key=lambda x: (round(x.real, 4), round(x.imag, 4)))

def _fmt_c(z):
    z = complex(z)
    return f"({z.real:.6f}{'+' if z.imag>=0 else '-'}{abs(z.imag):.6f}j)"

def diagnose_cubic(a, b, c, d, sqrt_iterations=10, cbrt_iterations=10):
    """Compute diagnostic values used by solve_cubic for given coefficients.
    Returns a dict with delta_0, delta_1, delta, sqrt_delta (pair), C_val (complex or None).
    """
    b_n = b / (-3.0 * a)
    c_n = c / (-3.0 * a)
    d_n = d / (-3.0 * a)

    delta_0 = b_n**2 + c_n
    delta_1 = 2.0 * b_n**3 + 3.0 * b_n * c_n + 3.0 * d_n
    delta = delta_1**2 - 4.0 * delta_0**3

    sqrt_pair = pade_sqrt(abs(float(delta.real)), iterations=sqrt_iterations)
    sqrt_delta = complex(0.0, sqrt_pair[0]) if delta.real < 0 else complex(*sqrt_pair)

    C_val = complex(*pade_cbrt((delta_1 + sqrt_delta) / 2.0, iterations=cbrt_iterations))

    return {
        'b_n': b_n,
        'c_n': c_n,
        'd_n': d_n,
        'delta_0': delta_0,
        'delta_1': delta_1,
        'delta': delta,
        'sqrt_delta': sqrt_delta,
        'C_val': C_val,
    }

def run_cubic_benchmark(num_tests=1000, sqrt_iterations=10, cbrt_iterations=10, report_failures=False, max_report=20):
    random.seed(7)
    pass_count = 0
    threshold = 1e-2
    max_err = 0.0

    failures = []

    for i in range(1, num_tests + 1):
        a = random.uniform(-10.0, 10.0)
        while abs(a) < 1e-3:
            a = random.uniform(-10.0, 10.0)
        b = random.uniform(-10.0, 10.0)
        c = random.uniform(-10.0, 10.0)
        d = random.uniform(-10.0, 10.0)

        my_roots = solve_cubic(a, b, c, d, sqrt_iterations=sqrt_iterations, cbrt_iterations=cbrt_iterations)
        std_roots = np.roots([a, b, c, d])

        my_roots_sorted = sort_roots(my_roots)
        std_roots_sorted = sort_roots(std_roots)

        case_err = 0.0
        for mr, sr in zip(my_roots_sorted, std_roots_sorted):
            err = abs(mr - sr)
            if err > case_err:
                case_err = err

        if case_err > max_err:
            max_err = case_err

        if case_err < threshold:
            pass_count += 1
        else:
            if report_failures:
                diag = diagnose_cubic(a, b, c, d, sqrt_iterations=sqrt_iterations, cbrt_iterations=cbrt_iterations)
                failures.append({
                    'index': i,
                    'a': a, 'b': b, 'c': c, 'd': d,
                    'err': case_err,
                    'my_roots': my_roots_sorted,
                    'std_roots': std_roots_sorted,
                    'diag': diag,
                })

    print(f"Tổng số test: {num_tests}")
    print(f"Số test khớp kết quả (sai số < {threshold}): {pass_count}/{num_tests}")
    print(f"Sai số lớn nhất ghi nhận: {max_err:.6f}")

    if report_failures and failures:
        print("\n--- Danh sách các test không đạt (giới hạn: {}/{} ) ---".format(min(max_report, len(failures)), len(failures)))
        header = f"{'#':>4}  {'a':>9}  {'b':>9}  {'c':>9}  {'d':>9}  {'max_err':>10}"
        print(header)
        print('-' * len(header))
        for f in failures[:max_report]:
            print(f"{f['index']:4d}  {f['a']:9.4f}  {f['b']:9.4f}  {f['c']:9.4f}  {f['d']:9.4f}  {f['err']:10.6e}")

        print('\n--- Chi tiết các nghiệm (my_vs_np.roots) và chẩn đoán cho các test đầu ---')
        for f in failures[:min(5, max_report)]:
            d = f['diag']
            print(f"\nTest #{f['index']}: a={f['a']:.6f}, b={f['b']:.6f}, c={f['c']:.6f}, d={f['d']:.6f}, err={f['err']:.6e}")
            print(f"  my_roots : {[ _fmt_c(r) for r in f['my_roots'] ]}")
            print(f"  np.roots : {[ _fmt_c(r) for r in f['std_roots'] ]}")
            print(f"  delta_0  : {d['delta_0']}")
            print(f"  delta_1  : {d['delta_1']}")
            print(f"  delta    : {d['delta']}")
            print(f"  sqrt_delta: {d['sqrt_delta']}")
            print(f"  C_val    : {d['C_val']}")

# Example quick run (smaller number to inspect failures):
run_cubic_benchmark(100000, sqrt_iterations=5, cbrt_iterations=5, report_failures=True, max_report=10)

Tổng số test: 100000
Số test khớp kết quả (sai số < 0.01): 100000/100000
Sai số lớn nhất ghi nhận: 0.001810
